# Single-threshold directional change

This first notebook builds the event ledger for one directional-change threshold. It deliberately stops before plotting, EDA, or regime modelling.

For each completed extreme-to-extreme trend:

$$\mathrm{TMV} = \frac{|P_{\mathrm{EXT},n} - P_{\mathrm{EXT},n-1}|}{P_{\mathrm{EXT},n-1}\,\theta}$$

$$T = t_{\mathrm{EXT},n} - t_{\mathrm{EXT},n-1}$$

$$R = \frac{\mathrm{TMV}\,\theta}{T}$$

The ending extreme is identified retrospectively. Therefore, TMV, T, and R are not available at `ext_end_idx`; they first become knowable at the later DC confirmation, `dcc_idx`. The notebook records that timestamp as `information_time`.

In [1]:
from dataclasses import dataclass, field
from enum import StrEnum
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

from dc_basics.core import parse_dc_events
from dc_basics.hmm import DistType, InferenceMode, create_simple_hmm_instance
from dc_basics.plotting import (
    plot_dc_indicator_distributions,
    plot_directional_change,
    plot_hmm_label_agreement,
    plot_hmm_model_event_strips,
    plot_hmm_model_label_shares,
    plot_hmm_model_price_comparison,
)

## Configuration

In [2]:
DATA_ROOT = Path(r"D:\data_dump\market_data\raw\FXPIG-Server\5")
SYMBOL = "US500.r"
DATA_PATH = DATA_ROOT / f"{SYMBOL}.parquet"

THETA = 0.003
N_PRICES = 5_000

print(f"Symbol: {SYMBOL}")
print(f"Data: {DATA_PATH}")
print(f"Threshold: {THETA:.4%}")
print(f"Chart window: {N_PRICES:,} prices")

Symbol: US500.r
Data: D:\data_dump\market_data\raw\FXPIG-Server\5\US500.r.parquet
Threshold: 0.3000%
Chart window: 5,000 prices


## Load and audit the bars

Directional-change parsing is path-dependent, so the price series must be chronological, unique, finite, and strictly positive. If `time` is stored as a column rather than the Parquet index, it is promoted to the index explicitly.

In [3]:
raw = pd.read_parquet(DATA_PATH)

raw = raw.sort_index()
prices = raw["close"].astype(np.float64)

print(f"Bars: {len(prices):,}")
print(f"Start: {prices.index[0]}")
print(f"End: {prices.index[-1]}")
print(f"Timezone: {prices.index.tz}")
raw.tail()

Bars: 228,362
Start: 2023-05-22 14:00:00+02:00
End: 2026-08-13 13:30:00+02:00
Timezone: pytz.FixedOffset(120)


,open,high,low,close,tick_volume,spread
date,,,,,,
2026-08-13 13:10:00+02:00,7765.89,7766.64,7764.52,7766.40,185.0,41
2026-08-13 13:15:00+02:00,7766.52,7766.89,7763.90,7765.77,241.0,41
2026-08-13 13:20:00+02:00,7765.77,7768.27,7764.77,7767.77,183.0,41
2026-08-13 13:25:00+02:00,7767.77,7769.02,7767.77,7768.40,153.0,41
2026-08-13 13:30:00+02:00,7768.52,7768.77,7767.27,7768.39,216.0,41


## Calculate completed directional-change events

`parse_dc_events` returns one row per completed trend. `event_id` gives each event a stable notebook identifier. `information_time` is deliberately copied from `dcc_idx`, because that confirmation bar is when the ending extreme and its completed TMV, T, and R become knowable.

In [4]:
events = parse_dc_events(prices=prices, theta=THETA).copy()

if events.empty:
    raise ValueError("No completed DC events were found. Check the data and threshold.")

events.insert(0, "event_id", np.arange(1, len(events) + 1, dtype=np.int64))
events.insert(1, "threshold", THETA)
events.insert(2, "information_time", events["dcc_idx"])

event_columns = [
    "event_id", "threshold", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "dcc_idx", "dcc_price",
    "information_time", "tmv", "t", "r",
]

print(f"Completed events: {len(events):,}")
events[event_columns].tail(10)

Completed events: 3,649


,event_id,threshold,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,dcc_idx,dcc_price,information_time,tmv,t,r
3639,3640,0.003,down,2026-08-05 18:55:00+02:00,7800.92,2026-08-06 22:25:00+02:00,7710.47,2026-08-07 16:05:00+02:00,7734.12,2026-08-07 16:05:00+02:00,3.864929,318,0.000036
3640,3641,0.003,up,2026-08-06 22:25:00+02:00,7710.47,2026-08-07 17:50:00+02:00,7760.25,2026-08-07 19:00:00+02:00,7734.87,2026-08-07 19:00:00+02:00,2.152052,221,0.000029
3641,3642,0.003,down,2026-08-07 17:50:00+02:00,7760.25,2026-08-07 19:10:00+02:00,7728.37,2026-08-07 19:20:00+02:00,7751.62,2026-08-07 19:20:00+02:00,1.369372,16,0.000257
3642,3643,0.003,up,2026-08-07 19:10:00+02:00,7728.37,2026-08-07 20:25:00+02:00,7768.12,2026-08-07 23:30:00+02:00,7744.37,2026-08-07 23:30:00+02:00,1.714462,15,0.000343
3643,3644,0.003,down,2026-08-07 20:25:00+02:00,7768.12,2026-08-07 23:40:00+02:00,7739.87,2026-08-10 06:15:00+02:00,7763.24,2026-08-10 06:15:00+02:00,1.212220,39,0.000093
3644,3645,0.003,up,2026-08-07 23:40:00+02:00,7739.87,2026-08-10 19:45:00+02:00,7780.87,2026-08-10 21:35:00+02:00,7757.37,2026-08-10 21:35:00+02:00,1.765749,229,0.000023
3645,3646,0.003,down,2026-08-10 19:45:00+02:00,7780.87,2026-08-10 21:40:00+02:00,7752.12,2026-08-11 16:00:00+02:00,7779.72,2026-08-11 16:00:00+02:00,1.231653,23,0.000161
3646,3647,0.003,up,2026-08-10 21:40:00+02:00,7752.12,2026-08-11 16:00:00+02:00,7779.72,2026-08-11 20:15:00+02:00,7755.72,2026-08-11 20:15:00+02:00,1.186772,208,0.000017
3647,3648,0.003,down,2026-08-11 16:00:00+02:00,7779.72,2026-08-11 23:25:00+02:00,7726.72,2026-08-12 14:00:00+02:00,7750.07,2026-08-12 14:00:00+02:00,2.270862,89,0.000077
3648,3649,0.003,up,2026-08-11 23:25:00+02:00,7726.72,2026-08-12 18:25:00+02:00,7777.07,2026-08-12 19:00:00+02:00,7750.07,2026-08-12 19:00:00+02:00,2.172116,216,0.000030


## Visualize the last N prices

The close is shown with completed extreme-to-extreme trends and their later DC confirmations. A turning-point marker describes where an extreme occurred; its hover text shows when that extreme became knowable. The final unfinished event is intentionally absent because `parse_dc_events` emits completed events only.

In [5]:
fig = plot_directional_change(prices=prices, events=events, n_prices=N_PRICES, title=f"{SYMBOL} | theta={THETA:.4%}",)
fig.show()

## Descriptive EDA of TMV, T, and R

The mean answers the average question, but these event indicators are typically right-skewed. Median and tail quantiles are reported beside it so a small number of extreme events cannot silently determine the interpretation. These are descriptive event-level results: consecutive DC events are path-dependent, so the rows must not be treated as independent observations for significance testing.

In [6]:
INDICATORS = ["tmv", "t", "r"]
QUANTILES = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]

overall_summary = events[INDICATORS].describe(percentiles=QUANTILES).T
overall_summary.index.name = "indicator"
overall_summary.style.format(precision=6)

,count,mean,std,min,10%,25%,50%,75%,90%,95%,99%,max
indicator,,,,,,,,,,,,
tmv,3649.000000,2.408017,1.557137,1.000390,1.132111,1.380040,1.922709,2.908315,4.224411,5.240841,8.070557,21.481858
t,3649.000000,62.521787,90.223514,1.000000,5.000000,11.000000,27.000000,74.000000,178.000000,239.600000,410.600000,1100.000000
r,3649.000000,0.000522,0.000936,0.000009,0.000045,0.000088,0.000223,0.000516,0.001257,0.002052,0.004731,0.011954


### Direction-conditioned averages

The event count must be read with every directional comparison. A difference between up and down averages is descriptive here; it is not yet evidence of a stable or tradable asymmetry.

In [7]:
direction_summary = events.groupby("direction", sort=False).agg(
    event_count=("event_id", "size"),
    tmv_mean=("tmv", "mean"),
    tmv_median=("tmv", "median"),
    t_mean=("t", "mean"),
    t_median=("t", "median"),
    r_mean=("r", "mean"),
    r_median=("r", "median"),
)
direction_summary.style.format({
    "event_count": "{:,}",
    "tmv_mean": "{:.6f}",
    "tmv_median": "{:.6f}",
    "t_mean": "{:.2f}",
    "t_median": "{:.2f}",
    "r_mean": "{:.6e}",
    "r_median": "{:.6e}",
})

,event_count,tmv_mean,tmv_median,t_mean,t_median,r_mean,r_median
direction,,,,,,,
up,"1,825",2.476115,1.946773,72.94,29.00,5.145303e-04,2.051897e-04
down,"1,824",2.339882,1.904402,52.10,25.50,5.294216e-04,2.407503e-04


### Direction-conditioned distributions

Each histogram is normalized within direction, so up and down shapes remain comparable even if their event counts differ. The initial x-axis view ends at the full-sample 99th percentile; Plotly's **Autoscale** button reveals all tail observations.

In [8]:
distribution_fig = plot_dc_indicator_distributions(events)
distribution_fig.show()

## Find events whose extremes are consecutive observations

`T = 1` means the starting and ending extremes occupy adjacent rows in the stored close series. `elapsed_minutes` checks whether those observations are also one 5-minute period apart in wall-clock time. `confirmation_delay_bars` measures how many subsequent observations were needed to confirm the ending extreme.

In [9]:
consecutive_events = events.loc[events["t"] == 1].copy()
consecutive_events["elapsed_minutes"] = (consecutive_events["ext_end_idx"] - consecutive_events["ext_start_idx"]).dt.total_seconds() / 60.0
consecutive_events["observed_move_pct"] = (consecutive_events["ext_end_price"] / consecutive_events["ext_start_price"] - 1.0) * 100.0
consecutive_events["confirmation_delay_bars"] = (consecutive_events["dcc_pos"] - consecutive_events["ext_end_pos"])

consecutive_columns = [
    "event_id", "direction", "ext_start_idx", "ext_start_price",
    "ext_end_idx", "ext_end_price", "elapsed_minutes", "observed_move_pct",
    "tmv", "r", "dcc_idx", "confirmation_delay_bars",
]

print(f"Events with T=1: {len(consecutive_events):,} of {len(events):,}")
consecutive_events[consecutive_columns]

Events with T=1: 59 of 3,649


,event_id,direction,ext_start_idx,ext_start_price,ext_end_idx,ext_end_price,elapsed_minutes,observed_move_pct,tmv,r,dcc_idx,confirmation_delay_bars
617,618,down,2024-02-02 19:35:00+02:00,4960.44,2024-02-02 19:40:00+02:00,4941.44,5.0,-0.383031,1.276768,0.003830,2024-02-02 20:15:00+02:00,7
691,692,down,2024-03-20 21:25:00+02:00,5202.49,2024-03-20 21:30:00+02:00,5186.14,5.0,-0.314273,1.047575,0.003143,2024-03-20 21:40:00+02:00,2
923,924,down,2024-07-19 00:45:00+02:00,5562.82,2024-07-19 00:50:00+02:00,5544.77,5.0,-0.324476,1.081586,0.003245,2024-07-19 03:05:00+02:00,15
992,993,up,2024-08-02 18:30:00+02:00,5358.32,2024-08-02 18:35:00+02:00,5381.32,5.0,0.429239,1.430797,0.004292,2024-08-02 18:55:00+02:00,4
1017,1018,down,2024-08-05 18:35:00+02:00,5174.47,2024-08-05 18:40:00+02:00,5158.92,5.0,-0.300514,1.001713,0.003005,2024-08-05 18:45:00+02:00,1
1018,1019,up,2024-08-05 18:40:00+02:00,5158.92,2024-08-05 18:45:00+02:00,5186.54,5.0,0.535383,1.784611,0.005354,2024-08-05 18:55:00+02:00,2
1020,1021,up,2024-08-05 18:55:00+02:00,5169.32,2024-08-05 19:00:00+02:00,5210.82,5.0,0.802814,2.676045,0.008028,2024-08-05 19:05:00+02:00,1
1027,1028,down,2024-08-06 01:05:00+02:00,5218.12,2024-08-06 01:10:00+02:00,5202.37,5.0,-0.301833,1.006109,0.003018,2024-08-06 01:55:00+02:00,9
1042,1043,up,2024-08-07 00:15:00+02:00,5275.22,2024-08-07 00:20:00+02:00,5292.97,5.0,0.336479,1.121596,0.003365,2024-08-07 00:40:00+02:00,4
1671,1672,down,2025-03-05 22:00:00+02:00,5824.27,2025-03-05 22:05:00+02:00,5799.02,5.0,-0.433531,1.445102,0.004335,2025-03-05 22:45:00+02:00,8


### Inspect one consecutive-extreme event on the raw bars

Change `EVENT_ID` to inspect another row from the table above. The displayed path begins at the starting extreme and ends at the later confirmation bar.

In [10]:
if consecutive_events.empty:
    raise ValueError("No T=1 events are available at the current threshold.")

EVENT_ID = int(consecutive_events.iloc[-1]["event_id"])
selected_event = events.loc[events["event_id"] == EVENT_ID].iloc[0]

event_bars = raw.iloc[
    int(selected_event["ext_start_pos"]):int(selected_event["dcc_pos"]) + 1
].copy()
event_bars["dc_landmark"] = ""
event_bars.loc[selected_event["ext_start_idx"], "dc_landmark"] = "start EXT"
event_bars.loc[selected_event["ext_end_idx"], "dc_landmark"] = "end EXT"
event_bars.loc[selected_event["dcc_idx"], "dc_landmark"] = "DCC confirmation"

print(
    f"Event {EVENT_ID}: {selected_event['direction']} | "
    f"T={selected_event['t']} | TMV={selected_event['tmv']:.4f} | R={selected_event['r']:.3e}"
)
event_bars[["open", "high", "low", "close", "dc_landmark"]]

Event 3453: up | T=1 | TMV=1.0046 | R=3.014e-03


,open,high,low,close,dc_landmark
date,,,,,
2026-06-11 19:15:00+02:00,7301.02,7307.27,7292.02,7293.39,start EXT
2026-06-11 19:20:00+02:00,7292.77,7319.62,7292.40,7315.37,end EXT
2026-06-11 19:25:00+02:00,7315.87,7320.62,7299.87,7310.87,
2026-06-11 19:30:00+02:00,7311.12,7316.62,7300.12,7304.12,
2026-06-11 19:35:00+02:00,7304.37,7323.62,7304.12,7314.62,
2026-06-11 19:40:00+02:00,7314.62,7321.74,7308.37,7309.87,
2026-06-11 19:45:00+02:00,7309.99,7313.12,7301.62,7303.37,
2026-06-11 19:50:00+02:00,7303.50,7305.37,7293.87,7294.62,
2026-06-11 19:55:00+02:00,7295.87,7296.37,7274.12,7276.62,DCC confirmation


## Competing HMM emission distributions

We now compare six two-state HMMs while keeping the event series, chronological 80/20 split, causal filtering, restart count, and semantic state definition fixed.

The observation scale follows each distribution's support:

- Normal, Student-t, and Lambda use `log(R)` because they have real-valued support.
- LogNormal, Gamma, and Exponential use raw positive `R`.

For a fair likelihood comparison, models fitted to `log(R)` are converted back to the raw-`R` density scale with the Jacobian $-\log(R)$. Normal-on-log-R and LogNormal-on-raw-R should therefore score similarly; that pair is a useful implementation sanity check.

The primary comparison is chronological out-of-sample predictive log-likelihood. Label balance is descriptive, not a selection target. The sequence-order premium compares the real OOS ordering with shuffled versions of the same OOS values; a positive premium indicates that the transition dynamics explain ordering beyond merely partitioning the marginal tail.

In [11]:
class ObservationTransform(StrEnum):
    RAW_R = "raw_R"
    LOG_R = "log_R"


class CandidateName(StrEnum):
    NORMAL_LOG_R = "normal_log_R"
    STUDENTT_LOG_R = "studentt_log_R"
    LAMBDA_LOG_R = "lambda_log_R"
    LOGNORMAL_RAW_R = "lognormal_raw_R"
    GAMMA_RAW_R = "gamma_raw_R"
    EXPONENTIAL_RAW_R = "exponential_raw_R"


@dataclass(frozen=True)
class HmmCandidateSpec:
    name: CandidateName
    distribution: DistType
    transform: ObservationTransform
    kwargs: dict[str, object] = field(default_factory=dict)


N_STATES = 2
TRAIN_FRACTION = 0.80
RANDOM_STATE = 100
MAX_ITER = 500
N_RESTARTS = 5
N_SEQUENCE_SHUFFLES = 100
COMPARISON_PRICE_BARS = 20_000

CANDIDATES = (
    HmmCandidateSpec(CandidateName.NORMAL_LOG_R, DistType.NORMAL, ObservationTransform.LOG_R,
                     {"covariance_type": "diag", "min_cov": 1e-6}),
    HmmCandidateSpec(CandidateName.STUDENTT_LOG_R, DistType.STUDENTT, ObservationTransform.LOG_R,
                     {"covariance_type": "diag", "min_cov": 1e-6, "dofs": 5}),
    HmmCandidateSpec(CandidateName.LAMBDA_LOG_R, DistType.LAMDA, ObservationTransform.LOG_R),
    HmmCandidateSpec(CandidateName.LOGNORMAL_RAW_R, DistType.LOGNORMAL, ObservationTransform.RAW_R,
                     {"covariance_type": "diag", "min_cov": 1e-12}),
    HmmCandidateSpec(CandidateName.GAMMA_RAW_R, DistType.GAMMA, ObservationTransform.RAW_R),
    HmmCandidateSpec(CandidateName.EXPONENTIAL_RAW_R, DistType.EXPONENTIAL, ObservationTransform.RAW_R),
)

In [12]:
comparison_events = events.copy().sort_values("information_time").reset_index(drop=True)
if comparison_events["r"].isna().any() or not np.isfinite(comparison_events["r"]).all():
    raise ValueError("R contains NaN or infinite observations.")
if (comparison_events["r"] <= 0).any():
    raise ValueError("All candidate distributions require strictly positive R before transformation.")

comparison_events["log_r"] = np.log(comparison_events["r"])
r_all = comparison_events[["r"]].to_numpy(dtype=float)
log_r_all = comparison_events[["log_r"]].to_numpy(dtype=float)
train_event_count = int(len(comparison_events) * TRAIN_FRACTION)
if train_event_count < N_STATES or train_event_count >= len(comparison_events):
    raise ValueError("The chronological split leaves too few train or OOS events.")

event_sample = np.where(np.arange(len(comparison_events)) < train_event_count, "train", "oos")
print(f"Events: {len(comparison_events):,} | train: {train_event_count:,} | OOS: {len(comparison_events) - train_event_count:,}")
print(f"Train ends: {comparison_events.loc[train_event_count - 1, 'information_time']}")
print(f"OOS starts: {comparison_events.loc[train_event_count, 'information_time']}")

Events: 3,649 | train: 2,919 | OOS: 730
Train ends: 2026-02-02 05:10:00+02:00
OOS starts: 2026-02-02 08:40:00+02:00


In [13]:
def observations_for(spec: HmmCandidateSpec) -> np.ndarray:
    return log_r_all if spec.transform == ObservationTransform.LOG_R else r_all


def raw_r_predictive_loglik(model, observations: np.ndarray, raw_r: np.ndarray, transform: ObservationTransform) -> np.ndarray:
    scores = model.compute_per_bar_predictive_loglik(observations)
    if transform == ObservationTransform.LOG_R:
        scores = scores - np.log(raw_r.reshape(-1))
    return scores


def weighted_state_centers(probabilities: np.ndarray, raw_r: np.ndarray) -> np.ndarray:
    weights = probabilities.sum(axis=0)
    if np.any(weights <= 0):
        raise ValueError("At least one fitted state has zero posterior mass in training.")
    return (probabilities * raw_r[:, None]).sum(axis=0) / weights


def flip_rate(labels: np.ndarray) -> float:
    return float(np.mean(labels[1:] != labels[:-1])) if len(labels) > 1 else np.nan

In [14]:
comparison_models = {}
comparison_label_frames = []
comparison_result_rows = []

for spec in CANDIDATES:
    model_name = spec.name.value
    try:
        observations = observations_for(spec)
        train_observations = observations[:train_event_count]
        model = create_simple_hmm_instance(
            dist_type=spec.distribution, n_states=N_STATES, random_state=RANDOM_STATE,
            max_iter=MAX_ITER, inference_mode=InferenceMode.FILTERING, n_restarts=N_RESTARTS, **spec.kwargs
        )
        model.fit(train_observations)

        filtered_probabilities = model.predict_proba(observations)
        fitted_centers = weighted_state_centers(
            filtered_probabilities[:train_event_count], r_all[:train_event_count].reshape(-1)
        )
        semantic_order = np.argsort(fitted_centers)
        semantic_probabilities = filtered_probabilities[:, semantic_order]
        semantic_centers = fitted_centers[semantic_order]
        semantic_state_index = np.argmax(semantic_probabilities, axis=1)
        semantic_labels = np.array(["slow_R", "fast_R"], dtype=object)[semantic_state_index]
        state_confidence = semantic_probabilities.max(axis=1)

        raw_scale_scores = raw_r_predictive_loglik(model, observations, r_all, spec.transform)
        train_scores = raw_scale_scores[:train_event_count]
        oos_scores = raw_scale_scores[train_event_count:]

        rng = np.random.default_rng(RANDOM_STATE)
        shuffled_oos_means = []
        for _ in range(N_SEQUENCE_SHUFFLES):
            permutation = rng.permutation(len(observations) - train_event_count)
            shuffled_observations = np.vstack(
                [train_observations, observations[train_event_count:][permutation]]
            )
            shuffled_raw_r = np.vstack([r_all[:train_event_count], r_all[train_event_count:][permutation]])
            shuffled_scores = raw_r_predictive_loglik(
                model, shuffled_observations, shuffled_raw_r, spec.transform
            )
            shuffled_oos_means.append(float(shuffled_scores[train_event_count:].mean()))

        transition_matrix = model.transition_prob()[np.ix_(semantic_order, semantic_order)]
        train_labels = semantic_labels[:train_event_count]
        oos_labels = semantic_labels[train_event_count:]
        train_fast_share = float(np.mean(train_labels == "fast_R"))
        oos_fast_share = float(np.mean(oos_labels == "fast_R"))
        collapsed = train_fast_share in {0.0, 1.0} or oos_fast_share in {0.0, 1.0}

        comparison_models[model_name] = model
        comparison_label_frames.append(pd.DataFrame({
            "event_id": comparison_events["event_id"].to_numpy(),
            "information_time": comparison_events["information_time"].to_numpy(),
            "sample": event_sample,
            "model": model_name,
            "state_name": semantic_labels,
            "state_probability": state_confidence,
            "p_fast_r": semantic_probabilities[:, 1],
        }))
        comparison_result_rows.append({
            "model": model_name,
            "distribution": spec.distribution.name,
            "observation": spec.transform.value,
            "status": "collapsed" if collapsed else "ok",
            "train_mean_raw_R_predictive_ll": float(train_scores.mean()),
            "oos_mean_raw_R_predictive_ll": float(oos_scores.mean()),
            "oos_p05_raw_R_predictive_ll": float(np.quantile(oos_scores, 0.05)),
            "oos_sequence_order_premium": float(oos_scores.mean() - np.median(shuffled_oos_means)),
            "slow_R_center": float(semantic_centers[0]),
            "fast_R_center": float(semantic_centers[1]),
            "center_ratio": float(semantic_centers[1] / semantic_centers[0]),
            "train_fast_share": train_fast_share,
            "oos_fast_share": oos_fast_share,
            "train_mean_confidence": float(state_confidence[:train_event_count].mean()),
            "oos_mean_confidence": float(state_confidence[train_event_count:].mean()),
            "train_flip_rate": flip_rate(train_labels),
            "oos_flip_rate": flip_rate(oos_labels),
            "slow_R_self_transition": float(transition_matrix[0, 0]),
            "fast_R_self_transition": float(transition_matrix[1, 1]),
            "error": "",
        })
    except Exception as exc:
        comparison_result_rows.append({
            "model": model_name, "distribution": spec.distribution.name,
            "observation": spec.transform.value, "status": "failed",
            "oos_mean_raw_R_predictive_ll": np.nan, "error": repr(exc)
        })

comparison_results = pd.DataFrame(comparison_result_rows)
ranked_results = comparison_results.sort_values(
    "oos_mean_raw_R_predictive_ll", ascending=False, na_position="last"
)
with pd.option_context("display.max_colwidth", None):
    display(ranked_results)

,model,distribution,observation,status,train_mean_raw_R_predictive_ll,oos_mean_raw_R_predictive_ll,oos_p05_raw_R_predictive_ll,oos_sequence_order_premium,slow_R_center,fast_R_center,center_ratio,train_fast_share,oos_fast_share,train_mean_confidence,oos_mean_confidence,train_flip_rate,oos_flip_rate,slow_R_self_transition,fast_R_self_transition,error
0,normal_log_R,NORMAL,log_R,ok,6.866463,6.947396,4.583297,5.993400e-02,0.000247,0.001685,6.808146,0.200069,0.109589,0.940939,0.929956,0.060315,0.079561,0.971338,0.893754,
4,gamma_raw_R,GAMMA,raw_R,ok,6.824788,6.914813,4.297911,6.241928e-02,0.000251,0.001911,7.602137,0.170949,0.101370,0.954637,0.952534,0.061001,0.096022,0.973293,0.875135,
2,lambda_log_R,LAMDA,log_R,ok,6.809727,6.898398,4.416869,3.224794e-02,0.000261,0.001466,5.622089,0.227133,0.117808,0.930745,0.915002,0.052776,0.060357,0.987144,0.960962,
1,studentt_log_R,STUDENTT,log_R,ok,6.813141,6.894374,4.515319,4.386459e-02,0.000248,0.001395,5.616707,0.243919,0.154795,0.915883,0.891313,0.067169,0.082305,0.977754,0.941496,
5,exponential_raw_R,EXPONENTIAL,raw_R,ok,6.812537,6.889269,4.251868,5.073467e-02,0.000253,0.001716,6.773732,0.187736,0.110959,0.941220,0.934777,0.052776,0.076818,0.979306,0.919037,
3,lognormal_raw_R,LOGNORMAL,raw_R,collapsed,-1.691436,-1.495597,-2.352205,1.110223e-15,0.000554,0.010189,18.379471,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000197,


### Reading the comparison

Prefer higher `oos_mean_raw_R_predictive_ll`, but require stability across chronological folds before choosing a production family. The fifth-percentile score exposes tail failures that the mean can hide. `oos_sequence_order_premium > 0` means the actual event ordering is more predictable than shuffled ordering; near zero means the result behaves mostly like a mixture model on the marginal R distribution.

The centers and labels are semantic summaries: `slow_R` is always the state with the lower probability-weighted training mean of raw R. They must not be used to select the winning model merely because one produces a visually pleasing orange share.

In [15]:
if not comparison_label_frames:
    raise RuntimeError("Every candidate HMM failed; inspect the error column above.")

comparison_event_labels = pd.concat(comparison_label_frames, ignore_index=True)
successful_models = comparison_event_labels["model"].drop_duplicates().tolist()
full_label_index = pd.MultiIndex.from_product(
    [successful_models, ["train", "oos"], ["slow_R", "fast_R"]],
    names=["model", "sample", "state_name"],
)
label_summary = (
    comparison_event_labels.groupby(["model", "sample", "state_name"]).size()
    .reindex(full_label_index, fill_value=0).rename("event_count").reset_index()
)
label_summary["share"] = label_summary["event_count"] / label_summary.groupby(
    ["model", "sample"]
)["event_count"].transform("sum")

display(label_summary)
plot_hmm_model_label_shares(label_summary)

,model,sample,state_name,event_count,share
0,normal_log_R,train,slow_R,2335,0.799931
1,normal_log_R,train,fast_R,584,0.200069
2,normal_log_R,oos,slow_R,650,0.890411
3,normal_log_R,oos,fast_R,80,0.109589
4,studentt_log_R,train,slow_R,2207,0.756081
5,studentt_log_R,train,fast_R,712,0.243919
6,studentt_log_R,oos,slow_R,617,0.845205
7,studentt_log_R,oos,fast_R,113,0.154795
8,lambda_log_R,train,slow_R,2256,0.772867
9,lambda_log_R,train,fast_R,663,0.227133


In [16]:
oos_label_matrix = (
    comparison_event_labels.loc[comparison_event_labels["sample"] == "oos"]
    .pivot(index="event_id", columns="model", values="state_name")
)
model_names = oos_label_matrix.columns.tolist()
raw_agreement = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
kappa_agreement = pd.DataFrame(index=model_names, columns=model_names, dtype=float)
for left_model in model_names:
    for right_model in model_names:
        left_labels = oos_label_matrix[left_model]
        right_labels = oos_label_matrix[right_model]
        valid = left_labels.notna() & right_labels.notna()
        raw_agreement.loc[left_model, right_model] = float(
            np.mean(left_labels.loc[valid].to_numpy() == right_labels.loc[valid].to_numpy())
        )
        kappa_agreement.loc[left_model, right_model] = cohen_kappa_score(
            left_labels.loc[valid], right_labels.loc[valid], labels=["slow_R", "fast_R"]
        )

display(raw_agreement.round(3))
plot_hmm_label_agreement(raw_agreement, title="OOS hard-label agreement")
display(kappa_agreement.round(3))
plot_hmm_label_agreement(kappa_agreement, title="OOS Cohen's kappa (base-rate adjusted)")

,exponential_raw_R,gamma_raw_R,lambda_log_R,lognormal_raw_R,normal_log_R,studentt_log_R
exponential_raw_R,1.000,0.974,0.925,0.889,0.968,0.921
gamma_raw_R,0.974,1.000,0.907,0.899,0.964,0.905
lambda_log_R,0.925,0.907,1.000,0.882,0.940,0.963
lognormal_raw_R,0.889,0.899,0.882,1.000,0.890,0.845
normal_log_R,0.968,0.964,0.940,0.890,1.000,0.938
studentt_log_R,0.921,0.905,0.963,0.845,0.938,1.000


,exponential_raw_R,gamma_raw_R,lambda_log_R,lognormal_raw_R,normal_log_R,studentt_log_R
exponential_raw_R,1.000,0.863,0.628,0.0,0.839,0.657
gamma_raw_R,0.863,1.000,0.523,0.0,0.811,0.580
lambda_log_R,0.628,0.523,1.000,0.0,0.701,0.843
lognormal_raw_R,0.000,0.000,0.000,NaN,0.000,0.000
normal_log_R,0.839,0.811,0.701,0.0,1.000,0.733
studentt_log_R,0.657,0.580,0.843,0.0,0.733,1.000


In [17]:
plot_hmm_model_event_strips(comparison_event_labels, sample="oos")

In [18]:
bar_model_labels = pd.DataFrame(index=prices.index)
for model_name, model_labels in comparison_event_labels.groupby("model", sort=False):
    event_updates = (
        model_labels.sort_values("information_time")
        .drop_duplicates("information_time", keep="last")
        .set_index("information_time")["state_name"]
    )
    bar_model_labels[model_name] = event_updates.reindex(prices.index).ffill()

plot_hmm_model_price_comparison(prices, bar_model_labels, n_prices=COMPARISON_PRICE_BARS)